## Distribution analysis

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, RobustScaler
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Read CSV files        
df1 = pd.read_csv('DATASETS/CDR-MLC/scale_0.001/Short/level_1.csv')
df2 = pd.read_csv('DATASETS/CDR-MLC/scale_0.001/Short/level_3.csv')

print("=" * 80)
print("DATASET INFORMATION")
print("=" * 80)
print(f"Dataset 1 (Long) shape: {df1.shape}")
print(f"Dataset 2 (Short) shape: {df2.shape}")
print(f"Dataset 1 columns: {len(df1.columns)}")
print(f"Dataset 2 columns: {len(df2.columns)}")

# Identify common columns
common_cols = list(set(df1.columns) & set(df2.columns))
print(f"\nNumber of common features: {len(common_cols)}")
print(f"Common features: {common_cols}")

# Separate numeric columns
numeric_cols = []
for col in common_cols:
    if pd.api.types.is_numeric_dtype(df1[col]) and pd.api.types.is_numeric_dtype(df2[col]):
        numeric_cols.append(col)

print(f"\nNumber of numeric features for scaling: {len(numeric_cols)}")
print(f"Numeric features: {numeric_cols}")

# Scale features
scaler = RobustScaler()

# Scale df1
df1_scaled = df1.copy()
if numeric_cols:
    df1_scaled[numeric_cols] = scaler.fit_transform(df1[numeric_cols])

# Scale df2
df2_scaled = df2.copy()
if numeric_cols:
    df2_scaled[numeric_cols] = scaler.fit_transform(df2[numeric_cols])

print("\n" + "=" * 80)
print("BASIC STATISTICS (Original Data)")
print("=" * 80)

for col in numeric_cols[:10]:  # Show first 10 features
    print(f"\nFeature: {col}")
    print(f"{'':<15} {'Dataset 1':<20} {'Dataset 2':<20}")
    print(f"{'Mean':<15} {df1[col].mean():<20.4f} {df2[col].mean():<20.4f}")
    print(f"{'Std':<15} {df1[col].std():<20.4f} {df2[col].std():<20.4f}")
    print(f"{'Min':<15} {df1[col].min():<20.4f} {df2[col].min():<20.4f}")
    print(f"{'Max':<15} {df1[col].max():<20.4f} {df2[col].max():<20.4f}")

print("\n" + "=" * 80)
print("BASIC STATISTICS (Scaled Data)")
print("=" * 80)

for col in numeric_cols[:10]:  # Show first 10 features
    print(f"\nFeature: {col}")
    print(f"{'':<15} {'Dataset 1':<20} {'Dataset 2':<20}")
    print(f"{'Mean':<15} {df1_scaled[col].mean():<20.4f} {df2_scaled[col].mean():<20.4f}")
    print(f"{'Std':<15} {df1_scaled[col].std():<20.4f} {df2_scaled[col].std():<20.4f}")

print("\n" + "=" * 80)
print("DISTRIBUTION COMPARISON TESTS")
print("=" * 80)

distribution_results = []
for col in numeric_cols:
    # Kolmogorov-Smirnov test for distribution similarity
    ks_stat, ks_pvalue = stats.ks_2samp(df1_scaled[col], df2_scaled[col])
    
    # T-test for mean difference
    t_stat, t_pvalue = stats.ttest_ind(df1_scaled[col], df2_scaled[col])
    
    # Calculate distribution overlap (simplified)
    hist1, bins = np.histogram(df1_scaled[col], bins=50, density=True)
    hist2, _ = np.histogram(df2_scaled[col], bins=bins, density=True)
    overlap = np.sum(np.minimum(hist1, hist2)) * (bins[1] - bins[0])
    
    distribution_results.append({
        'feature': col,
        'ks_statistic': ks_stat,
        'ks_pvalue': ks_pvalue,
        't_statistic': t_stat,
        't_pvalue': t_pvalue,
        'distribution_overlap': overlap
    })
    
    if col in numeric_cols[:5]:  # Show first 5 features
        print(f"\nFeature: {col}")
        print(f"KS Test: Statistic={ks_stat:.4f}, p-value={ks_pvalue:.4e}")
        print(f"T-Test: Statistic={t_stat:.4f}, p-value={t_pvalue:.4e}")
        print(f"Distribution Overlap: {overlap:.2%}")

print("\n" + "=" * 80)
print("SUMMARY OF SIGNIFICANT DIFFERENCES (p-value < 0.05)")
print("=" * 80)
# مرتب‌سازی نتایج بر اساس distribution_overlap (از کم به زیاد)
sorted_results = sorted(distribution_results, key=lambda x: x['distribution_overlap'])

significant_features = []
for result in sorted_results:
    if result['ks_pvalue'] < 0.05 or result['t_pvalue'] < 0.05:
        significant_features.append(result['feature'])
        print(f"{result['feature']}: KS-p={result['ks_pvalue']:.4e}, T-p={result['t_pvalue']:.4e}, Overlap={result['distribution_overlap']:.2%}")

print(f"\nTotal features with significant differences: {len(significant_features)}/{len(numeric_cols)}")

DATASET INFORMATION
Dataset 1 (Long) shape: (68079, 37)
Dataset 2 (Short) shape: (60482, 37)
Dataset 1 columns: 37
Dataset 2 columns: 37

Number of common features: 37
Common features: ['SrcPkts', 'TotPkts', 'Loss', 'DstLoad', 'PCRatio', 'Sum', 'sTtl', 'Min', 'Dur', 'pLoss', 'dMeanPktSz', 'DstLoss', 'SrcWin', 'DstPkts', 'DstRate', 'DstWin', 'Mean', 'SrcGap', 'TotBytes', 'label', 'SrcRate', 'SrcBytes', 'Load', 'DstRetra', 'Rate', 'pRetran', 'DstGap', 'StdDev', 'SrcLoad', 'TcpRtt', 'SrcRetra', 'SynAck', 'DstBytes', 'IdleTime', 'AckDat', 'SrcLoss', 'dTtl']

Number of numeric features for scaling: 36
Numeric features: ['SrcPkts', 'TotPkts', 'Loss', 'DstLoad', 'PCRatio', 'Sum', 'sTtl', 'Min', 'Dur', 'pLoss', 'dMeanPktSz', 'DstLoss', 'SrcWin', 'DstPkts', 'DstRate', 'DstWin', 'Mean', 'SrcGap', 'TotBytes', 'SrcRate', 'SrcBytes', 'Load', 'DstRetra', 'Rate', 'pRetran', 'DstGap', 'StdDev', 'SrcLoad', 'TcpRtt', 'SrcRetra', 'SynAck', 'DstBytes', 'IdleTime', 'AckDat', 'SrcLoss', 'dTtl']

BASIC STATI

## Trend analysis

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

def calculate_trend_score(window):
    """
    Calculate trend score for a data window
    """
    if len(window) < 2:
        return 0.0
    
    def normalized_global_slope(window):
        x = np.arange(len(window))
        slope, _ = np.polyfit(x, window, 1)
        max_slope = np.max(np.abs(np.diff(window))) if len(window) > 1 else 1
        return slope / max_slope if max_slope != 0 else 0
    
    def pairwise_slopes(window):
        return np.diff(window)
    
    def consistency_factor(P):
        if len(P) == 0:
            return 0.0
        first_sign = np.sign(P[0])
        if first_sign == 0:
            first_sign = 1
        same_sign_count = np.sum(np.sign(P) == first_sign)
        return same_sign_count / len(P)
    
    def contradiction_factor(G, P):
        if len(P) == 0:
            return 0.0
        global_sign = np.sign(G)
        if global_sign == 0:
            return 0.0
        contradictory_count = np.sum(np.sign(P) != global_sign)
        return contradictory_count / len(P)
    
    G = normalized_global_slope(window)
    P = pairwise_slopes(window)
    CF = consistency_factor(P)
    CD = contradiction_factor(G, P)
    
    G = np.clip(G, -1, 1)
    CF = np.clip(CF, 0, 1)
    CD = np.clip(CD, 0, 1)
    
    Trend_Score = (3*G) * (8*CF) * (2*(1 - CD))
    return np.clip(Trend_Score, -1, 1)

def analyze_trends(csv_files, window_size=10, num_batches=5, batch_size=None):
    """
    Analyze trends of features in multiple CSV files
    
    Parameters:
    -----------
    csv_files : list
        List of CSV file paths
    window_size : int
        Sliding window size for trend calculation
    num_batches : int
        Number of batches for analysis
    batch_size : int or None
        Size of each batch (if None, calculated automatically)
    """
    
    # Read CSV files
    print("📁 Reading CSV files...")
    dataframes = []
    for i, csv_file in enumerate(csv_files):
        df = pd.read_csv(csv_file)
        print(f"  File {i+1}: {csv_file}")
        print(f"    Data shape: {df.shape}")
        print(f"    Columns: {list(df.columns)}")
        dataframes.append(df)
    
    # Check column compatibility
    all_columns = [set(df.columns) for df in dataframes]
    common_columns = set.intersection(*all_columns)
    
    if len(common_columns) == 0:
        raise ValueError("❌ No common columns found between files!")
    
    print(f"\n✅ Common columns: {len(common_columns)} columns")
    print(f"   {list(common_columns)}")
    
    # Select only common numeric columns
    numeric_columns = []
    for col in common_columns:
        if all(pd.api.types.is_numeric_dtype(df[col]) for df in dataframes):
            numeric_columns.append(col)
    
    print(f"\n📊 Common numeric columns: {len(numeric_columns)} columns")
    
    # Extract numeric data
    all_data = []
    for i, df in enumerate(dataframes):
        df_numeric = df[numeric_columns].copy()
        all_data.append(df_numeric.values)
        print(f"  File {i+1}: {df_numeric.shape[0]} samples × {df_numeric.shape[1]} features")
    
    # Determine batch size
    min_samples = min(data.shape[0] for data in all_data)
    if batch_size is None:
        batch_size = min_samples // num_batches
        batch_size = max(batch_size, window_size * 2)
    
    print(f"\n⚙️  Analysis parameters:")
    print(f"   Window size: {window_size}")
    print(f"   Number of batches: {num_batches}")
    print(f"   Batch size: {batch_size}")
    print(f"   Total analyzed samples: {batch_size * num_batches}")
    
    # Scale the data
    print("\n🔧 Scaling data...")
    scalers = []
    scaled_data = []
    
    for i, data in enumerate(all_data):
        scaler = StandardScaler()
        data_scaled = scaler.fit_transform(data)
        scalers.append(scaler)
        scaled_data.append(data_scaled)
        print(f"  File {i+1}: Scaled")
    
    # Calculate trend for each batch
    print(f"\n📈 Calculating feature trends...")
    
    # Store results
    all_trend_scores = []
    feature_names = numeric_columns
    
    for batch_idx in range(num_batches):
        print(f"\n  Batch {batch_idx + 1}/{num_batches}:")
        
        batch_trends = []
        
        for file_idx, data in enumerate(scaled_data):
            print(f"    File {file_idx + 1}: ", end="")
            
            # Calculate batch start and end
            start_idx = batch_idx * batch_size
            end_idx = start_idx + batch_size
            
            if end_idx > len(data):
                end_idx = len(data)
                start_idx = max(0, end_idx - batch_size)
            
            batch_data = data[start_idx:end_idx]
            
            # Calculate trend for each feature
            file_trends = []
            for feature_idx in tqdm(range(data.shape[1]), desc="Features", leave=False):
                feature_data = batch_data[:, feature_idx]
                
                # Calculate trend with sliding window
                trend_scores = []
                for w_start in range(0, len(feature_data) - window_size + 1):
                    window = feature_data[w_start:w_start + window_size]
                    score = calculate_trend_score(window)
                    trend_scores.append(score)
                
                # Average trend score for this feature
                if trend_scores:
                    avg_trend = np.mean(trend_scores)
                else:
                    avg_trend = 0.0
                
                file_trends.append(avg_trend)
            
            batch_trends.append(file_trends)
            print(f"✅ ({len(batch_data)} samples)")
        
        all_trend_scores.append(batch_trends)
    
    # Analyze results
    print("\n" + "="*60)
    print("📊 Feature Trend Analysis Results")
    print("="*60)
    
    # Convert to numpy array for easier calculations
    all_trend_scores = np.array(all_trend_scores)  # Shape: (num_batches, num_files, num_features)
    
    # Calculate mean trend for each feature in each file
    mean_trends_per_file = np.mean(all_trend_scores, axis=0)  # Average over batches
    
    # Analysis for each file
    for file_idx in range(len(csv_files)):
        print(f"\n📁 File: {csv_files[file_idx]}")
        print("-" * 40)
        
        trends = mean_trends_per_file[file_idx]
        
        # Sort features by absolute trend value
        sorted_indices = np.argsort(np.abs(trends))[::-1]
        
        print("\n🎯 Top 10 features with strongest trend (positive or negative):")
        print("-" * 50)
        print(f"{'Rank':<6} {'Feature':<25} {'Trend':<10} {'Direction':<10}")
        print("-" * 50)
        
        for rank, idx in enumerate(sorted_indices[:10], 1):
            feature = feature_names[idx]
            trend = trends[idx]
            direction = "↑ Upward" if trend > 0 else "↓ Downward" if trend < 0 else "─ No trend"
            print(f"{rank:<6} {feature:<25} {trend:>8.4f} {direction:>10}")
    
    # Compare between files
    print("\n" + "="*60)
    print("🔄 Trend Comparison Between Files")
    print("="*60)
    
    if len(csv_files) == 2:
        # For two files, calculate trend difference
        diff_trends = mean_trends_per_file[0] - mean_trends_per_file[1]
        
        print("\n📊 Trend Difference (File 1 - File 2):")
        print("-" * 50)
        print(f"{'Rank':<6} {'Feature':<25} {'Trend Diff':<12} {'Explanation':<20}")
        print("-" * 50)
        
        # Sort by absolute difference
        diff_sorted_indices = np.argsort(np.abs(diff_trends))[::-1]
        
        for rank, idx in enumerate(diff_sorted_indices[:15], 1):
            feature = feature_names[idx]
            diff = diff_trends[idx]
            
            if diff > 0.1:
                explanation = "Stronger trend in File 1"
            elif diff < -0.1:
                explanation = "Stronger trend in File 2"
            elif abs(diff) < 0.05:
                explanation = "Similar trend"
            else:
                explanation = "Minor difference"
            
            print(f"{rank:<6} {feature:<25} {diff:>10.4f} {explanation:>20}")
    
    # Statistical summary
    print("\n" + "="*60)
    print("📈 Statistical Summary of Feature Trends")
    print("="*60)
    
    for file_idx, csv_file in enumerate(csv_files):
        trends = mean_trends_per_file[file_idx]
        
        print(f"\n📁 {csv_file}:")
        print(f"  Mean absolute trend: {np.mean(np.abs(trends)):.4f}")
        print(f"  Maximum positive trend: {np.max(trends):.4f}")
        print(f"  Maximum negative trend: {np.min(trends):.4f}")
        print(f"  Features with positive trend (>0.1): {np.sum(trends > 0.1)}")
        print(f"  Features with negative trend (<-0.1): {np.sum(trends < -0.1)}")
        print(f"  Features with neutral trend (|trend|≤0.1): {np.sum(np.abs(trends) <= 0.1)}")
    
    # Return results for further use
    results = {
        'feature_names': feature_names,
        'mean_trends': mean_trends_per_file,
        'all_trend_scores': all_trend_scores,
        'file_names': csv_files
    }
    
    return results

# Function for graphical visualization
def visualize_trends(results, top_n=20):
    """
    Visualize trend analysis results
    """
    try:
        import matplotlib.pyplot as plt
        import seaborn as sns
        
        sns.set_style("whitegrid")
        plt.figure(figsize=(15, 10))
        
        feature_names = results['feature_names']
        mean_trends = results['mean_trends']
        file_names = results['file_names']
        
        # For each file
        for file_idx, file_name in enumerate(file_names):
            trends = mean_trends[file_idx]
            
            # Select top_n features
            top_indices = np.argsort(np.abs(trends))[::-1][:top_n]
            top_features = [feature_names[i] for i in top_indices]
            top_trends = [trends[i] for i in top_indices]
            
            # Create colors based on trend direction
            colors = ['green' if t > 0 else 'red' for t in top_trends]
            
            plt.subplot(len(file_names), 1, file_idx + 1)
            bars = plt.barh(range(len(top_features)), top_trends, color=colors)
            plt.yticks(range(len(top_features)), top_features)
            plt.xlabel('Average Trend Score')
            plt.title(f'Features with Strongest Trend - {file_name}')
            plt.grid(True, alpha=0.3)
            
            # Add values on bars
            for bar, trend in zip(bars, top_trends):
                width = bar.get_width()
                label_x_pos = width if width >= 0 else width
                plt.text(label_x_pos, bar.get_y() + bar.get_height()/2,
                        f'{trend:.3f}', 
                        va='center', ha='left' if width >= 0 else 'right',
                        color='black', fontsize=9)
        
        plt.tight_layout()
        plt.show()
        
    except ImportError:
        print("⚠️ To visualize results, install matplotlib and seaborn:")
        print("   pip install matplotlib seaborn")

# Advanced analysis functions
def export_trend_analysis(results, output_file='trend_analysis_results.csv'):
    """
    Export trend analysis results to CSV
    """
    feature_names = results['feature_names']
    mean_trends = results['mean_trends']
    file_names = results['file_names']
    
    # Create DataFrame for export
    export_data = []
    for file_idx, file_name in enumerate(file_names):
        trends = mean_trends[file_idx]
        
        for feature_idx, feature in enumerate(feature_names):
            export_data.append({
                'File': file_name,
                'Feature': feature,
                'Trend_Score': trends[feature_idx],
                'Absolute_Trend': abs(trends[feature_idx]),
                'Direction': 'Upward' if trends[feature_idx] > 0 else 'Downward' if trends[feature_idx] < 0 else 'Neutral'
            })
    
    df_export = pd.DataFrame(export_data)
    
    # Sort by absolute trend score
    df_export = df_export.sort_values(['File', 'Absolute_Trend'], ascending=[True, False])
    
    # Save to CSV
    df_export.to_csv(output_file, index=False)
    print(f"✅ Results exported to: {output_file}")
    
    return df_export

def analyze_trend_correlation(results):
    """
    Analyze correlation between trends in different files
    """
    mean_trends = results['mean_trends']
    feature_names = results['feature_names']
    file_names = results['file_names']
    
    if len(file_names) >= 2:
        print("\n" + "="*60)
        print("🔗 Trend Correlation Analysis")
        print("="*60)
        
        # Calculate correlation matrix
        trend_matrix = np.array(mean_trends).T  # Transpose: features × files
        correlation_matrix = np.corrcoef(trend_matrix, rowvar=False)
        
        print("\n📊 Correlation between files:")
        for i in range(len(file_names)):
            for j in range(i+1, len(file_names)):
                corr = correlation_matrix[i, j]
                print(f"  {file_names[i]} ↔ {file_names[j]}: {corr:.4f}")
        
        # Find features with similar trends across files
        print("\n🎯 Features with consistent trends across all files:")
        consistent_features = []
        
        for feature_idx, feature in enumerate(feature_names):
            trends = trend_matrix[feature_idx]
            
            # Check if all trends have same sign
            if np.all(trends > 0) or np.all(trends < 0):
                consistent_features.append({
                    'feature': feature,
                    'trends': trends,
                    'mean_abs_trend': np.mean(np.abs(trends))
                })
        
        # Sort by mean absolute trend
        consistent_features.sort(key=lambda x: x['mean_abs_trend'], reverse=True)
        
        print(f"  Found {len(consistent_features)} features with consistent direction")
        if consistent_features:
            print("  Top 5 consistent features:")
            for i, cf in enumerate(consistent_features[:5], 1):
                directions = ['↑' if t > 0 else '↓' for t in cf['trends']]
                print(f"    {i}. {cf['feature']}: {directions} (avg: {cf['mean_abs_trend']:.4f})")
    
    return correlation_matrix if len(file_names) >= 2 else None

# Example usage
if __name__ == "__main__":
    # Specify CSV files
    csv_files = [
        'DATASETS/CDR-MLC/scale_5/Long/CDR-MLC-Shuffle.csv',  
        'DATASETS/CDR-MLC/scale_5/Short/CDR-MLC-Shuffle.csv'   
    ]
    
    # Configurable parameters
    WINDOW_SIZE = 10      # Sliding window size
    NUM_BATCHES = 5       # Number of batches
    BATCH_SIZE = 1000     # Batch size (if None, calculated automatically)
    
    # Run analysis
    results = analyze_trends(
        csv_files=csv_files,
        window_size=WINDOW_SIZE,
        num_batches=NUM_BATCHES,
        batch_size=BATCH_SIZE
    )
    
    # Additional analyses
    export_trend_analysis(results, 'trend_results.csv')
    analyze_trend_correlation(results)
    
    # Visualize results (optional)
    # visualize_trends(results, top_n=15)
    
    print("\n✅ Trend analysis completed successfully!")

📁 Reading CSV files...
  File 1: DATASETS/CDR-MLC/scale_5/Long/CDR-MLC-Shuffle.csv
    Data shape: (216201, 40)
    Columns: ['pRetran', 'sMeanPktSz', 'SrcRetra', 'PCRatio', 'SrcWin', 'SrcLoss', 'DstRate', 'SrcLoad', 'Load', 'DstLoad', 'TcpRtt', 'Sum', 'AckDat', 'dTtl', 'Min', 'Max', 'pLoss', 'DstLoss', 'Loss', 'StdDev', 'Rate', 'SrcRate', 'IdleTime', 'Dur', 'SrcPkts', 'SrcGap', 'DstBytes', 'DstGap', 'sTtl', 'DstWin', 'TotPkts', 'DstPkts', 'Mean', 'SrcBytes', 'TotBytes', 'dMeanPktSz', 'DstRetra', 'SynAck', 'label', 'index_in_label']
  File 2: DATASETS/CDR-MLC/scale_5/Short/CDR-MLC-Shuffle.csv
    Data shape: (4792, 41)
    Columns: ['pRetran', 'sMeanPktSz', 'SrcRetra', 'PCRatio', 'SrcWin', 'SrcLoss', 'DstRate', 'SrcLoad', 'Load', 'DstLoad', 'TcpRtt', 'Sum', 'AckDat', 'dTtl', 'Min', 'Max', 'pLoss', 'DstLoss', 'Loss', 'StdDev', 'Rate', 'SrcRate', 'IdleTime', 'Dur', 'SrcPkts', 'SrcGap', 'DstBytes', 'DstGap', 'sTtl', 'DstWin', 'TotPkts', 'DstPkts', 'Mean', 'SrcBytes', 'TotBytes', 'dMeanPkt

✅ (1000 samples)
    File 2: 

✅ (1000 samples)

  Batch 2/5:
    File 1: 

✅ (1000 samples)
    File 2: 

✅ (1000 samples)

  Batch 3/5:
    File 1: 

...

✅ (1000 samples)

  Batch 4/5:
    File 1: 

✅ (1000 samples)
    File 2: 

✅ (1000 samples)

  Batch 5/5:
    File 1: 

✅ (1000 samples)
    File 2: 

✅ (1000 samples)

📊 Feature Trend Analysis Results

📁 File: DATASETS/CDR-MLC/scale_5/Long/CDR-MLC-Shuffle.csv
----------------------------------------

🎯 Top 10 features with strongest trend (positive or negative):
--------------------------------------------------
Rank   Feature                   Trend      Direction 
--------------------------------------------------
1      TotBytes                    0.0480   ↑ Upward
2      SrcBytes                    0.0412   ↑ Upward
3      SrcPkts                     0.0398   ↑ Upward
4      Load                        0.0378   ↑ Upward
5      SrcLoad                     0.0362   ↑ Upward
6      TotPkts                     0.0349   ↑ Upward
7      SrcRate                     0.0340   ↑ Upward
8      IdleTime                    0.0314   ↑ Upward
9      Rate                        0.0273   ↑ Upward
10     DstPkts                     0.0247   ↑ Upward

📁 File: DATASETS/CDR-MLC/scale_5/Short/CDR-MLC-Shuffle.csv
--------------------------------------